<a href="https://www.kaggle.com/code/ab0y04/skin-lesion-imagenet?scriptVersionId=342885929" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD (new session) + STAGE 16 FOLD 3: MOBILENETV2 + RESNET50 =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # MUST precede tensorflow import
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42  # fixed globally across ALL folds and ALL architectures
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, recall_score, confusion_matrix
print("Imports ready")

FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES = 6
IMG_SIZE, BATCH_SIZE = 224, 32
N_FOLDS, CURRENT_FOLD = 5, 3
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print(f"Config loaded, CURRENT_FOLD = {CURRENT_FOLD}")

CV_ASSIGN_PATH = '/kaggle/input/datasets/ab0y04/cvfoldassignments/cv_fold_assignments.csv'
assert os.path.exists(CV_ASSIGN_PATH), f"STOP: file not found at {CV_ASSIGN_PATH}, check the dataset is attached"
cv_assignments = pd.read_csv(CV_ASSIGN_PATH)
print(f"Loaded cv_fold_assignments.csv: {len(cv_assignments):,} rows (expect 23,836)")
assert len(cv_assignments) == 23836, "STOP: row count mismatch"

def build_fold_split(cv_assignments, fold_num, seed=42):
    test_df = cv_assignments[cv_assignments['fold'] == fold_num].reset_index(drop=True)
    remaining = cv_assignments[cv_assignments['fold'] != fold_num].reset_index(drop=True)
    remaining = remaining.copy()
    fallback = pd.Series('unlinked_' + remaining.index.astype(str), index=remaining.index)
    remaining['_split_key'] = remaining['group_id'].fillna(fallback)
    groups = remaining.groupby('_split_key')['label'].first().reset_index()
    tr_groups, va_groups = train_test_split(groups, test_size=0.15, stratify=groups['label'], random_state=seed)
    train_df = remaining[remaining['_split_key'].isin(tr_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    val_df = remaining[remaining['_split_key'].isin(va_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = build_fold_split(cv_assignments, CURRENT_FOLD, seed=SEED)
print(f"\n===== FOLD {CURRENT_FOLD} SPLIT =====")
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print("(expect to match this fold's prior runs exactly: 16,230 / 2,852 / 4,754)")

test_groups = set(test_df['group_id'].dropna())
train_groups = set(train_df['group_id'].dropna())
val_groups = set(val_df['group_id'].dropna())
assert test_groups.isdisjoint(train_groups) and test_groups.isdisjoint(val_groups) and train_groups.isdisjoint(val_groups), \
    f"STOP: FOLD {CURRENT_FOLD} LEAKAGE detected"
print(f"Fold {CURRENT_FOLD} leakage check: PASS")

cls = np.array(FINAL_CLASSES)
cw = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print(f"Fold {CURRENT_FOLD} class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})
print("(expect: bcc 1.165, df 17.796, nevus 0.308)")

def make_fold_gens(preprocess_fn):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df, shuffle=False, **common)
    return tr, va, te

def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                          Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def macro_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp)>0 else np.nan)
    return np.nanmean(specs)

print(f"\n===== Fold {CURRENT_FOLD} setup verified. Training MobileNetV2 + ResNet50. =====\n")

FOLD1_ACC = {'mob': 0.6720463480240016, 'res': 0.7428098489551004}
FOLD2_ACC = {'mob': 0.608874281018899, 'res': 0.7191865242399342}
fold3_results = []

ARCHS = [('mob', MobileNetV2, mob_pre), ('res', ResNet50, res_pre)]
for arch_code, arch_class, prep_fn in ARCHS:
    try:
        print(f"{'='*60}\nFOLD {CURRENT_FOLD}: {arch_code}\n{'='*60}")
        tr, va, te = make_fold_gens(prep_fn)
        print("class_indices:", te.class_indices)

        model, base = build_pretrained(arch_class)
        log_file = f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_log.csv'

        base.trainable = False
        model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
        model.fit(tr, validation_data=va, epochs=10, callbacks=[CSVLogger(log_file, append=False)], verbose=1)
        print(f"Phase 1 sanity ({arch_code}):", model.evaluate(te, verbose=0))

        base.trainable = True
        model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
        cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
               ModelCheckpoint(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}.keras', monitor='val_accuracy', save_best_only=True),
               CSVLogger(log_file, append=True)]
        model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)

        y_true = np.asarray(te.classes)
        y_prob = model.predict(te, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)
        np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_preds_{arch_code}.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)

        reloaded = load_model(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}.keras')
        verify_acc = reloaded.evaluate(te, verbose=0)[1]
        live_acc = accuracy_score(y_true, y_pred)
        print(f"Checkpoint verify: reloaded acc {verify_acc:.4f} vs live acc {live_acc:.4f}  match={abs(verify_acc-live_acc)<1e-3}")
        del reloaded

        result_row = dict(fold=CURRENT_FOLD, arch=arch_code, accuracy=live_acc,
            macro_f1=f1_score(y_true,y_pred,average='macro'),
            macro_auc=roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr'),
            macro_sensitivity=recall_score(y_true,y_pred,average='macro'),
            macro_specificity=macro_specificity(y_true,y_pred,NUM_CLASSES), n_test=len(y_true))
        fold3_results.append(result_row)

        prior_mean = np.mean([FOLD1_ACC[arch_code], FOLD2_ACC[arch_code]])
        deviation = abs(live_acc - prior_mean) * 100
        flag = "  <-- FLAG: deviates >5pp from folds 1-2 mean" if deviation > 5 else "  (within normal range)"
        print(f"\nFold {CURRENT_FOLD} vs Folds 1-2 mean: {live_acc:.4f} vs {prior_mean:.4f}, deviation {deviation:.1f}pp{flag}")
        print(f"RESULT: {result_row}")
        pd.DataFrame([result_row]).to_csv(f'/kaggle/working/cv_f{CURRENT_FOLD}_{arch_code}_result.csv', index=False)
        del model, base; gc.collect(); tf.keras.backend.clear_session()
    except Exception as e:
        print(f"!!! FOLD {CURRENT_FOLD} {arch_code} FAILED: {type(e).__name__}: {e}")
        import traceback; traceback.print_exc()
        gc.collect(); tf.keras.backend.clear_session()

print(f"\n{'='*60}\nFOLD {CURRENT_FOLD} FINAL PAIR SUMMARY (mob + res)\n{'='*60}")
for r in fold3_results:
    print(f"{r['arch']:8} acc={r['accuracy']:.4f}  macro_f1={r['macro_f1']:.4f}  macro_auc={r['macro_auc']:.4f}")
print(f"\nCompleted this run: {len(fold3_results)}/2. Fold {CURRENT_FOLD} will be complete after this (4/4).")
print(">>> DOWNLOAD BOTH cv_f3_mob_result.csv AND cv_f3_res_result.csv NOW. <<<")

2026-08-17 02:04:09.694778: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786932249.911852      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786932249.991308      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786932250.516640      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786932250.516682      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786932250.516685      25 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Imports ready
Config loaded, CURRENT_FOLD = 3
Loaded cv_fold_assignments.csv: 23,836 rows (expect 23,836)

===== FOLD 3 SPLIT =====
Train 16,230 | Val 2,852 | Test 4,754
(expect to match this fold's prior runs exactly: 16,230 / 2,852 / 4,754)
Fold 3 leakage check: PASS
Fold 3 class_weight: {np.str_('bcc'): np.float64(1.165), np.str_('bkl'): np.float64(1.527), np.str_('df'): np.float64(17.796), np.str_('melanoma'): np.float64(0.892), np.str_('nevus'): np.float64(0.308), np.str_('vasc'): np.float64(15.282)}
(expect: bcc 1.165, df 17.796, nevus 0.308)

===== Fold 3 setup verified. Training MobileNetV2 + ResNet50. =====

FOLD 3: mob
Found 16230 validated image filenames belonging to 6 classes.
Found 2852 validated image filenames belonging to 6 classes.
Found 4754 validated image filenames belonging to 6 classes.
class_indices: {'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}


I0000 00:00:1786932308.921220      25 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786932308.927088      25 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 [==============================] - 1s 0us/step
Epoch 1/10


I0000 00:00:1786932316.807775      69 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1786932319.116977      67 service.cc:152] XLA service 0x7fe81c36b6c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1786932319.117013      67 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1786932319.117017      67 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1786932319.386177      67 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


508/508 [==============================] - 523s 1s/step - loss: 1.4794 - accuracy: 0.4558 - val_loss: 1.2030 - val_accuracy: 0.5480
Epoch 2/10
508/508 [==============================] - 359s 706ms/step - loss: 1.1903 - accuracy: 0.5112 - val_loss: 1.1042 - val_accuracy: 0.6087
Epoch 3/10
508/508 [==============================] - 353s 695ms/step - loss: 1.1120 - accuracy: 0.5329 - val_loss: 1.6147 - val_accuracy: 0.4060
Epoch 4/10
508/508 [==============================] - 349s 687ms/step - loss: 1.0423 - accuracy: 0.5564 - val_loss: 1.2727 - val_accuracy: 0.5028
Epoch 5/10
508/508 [==============================] - 346s 682ms/step - loss: 1.0043 - accuracy: 0.5633 - val_loss: 1.0126 - val_accuracy: 0.6241
Epoch 6/10
508/508 [==============================] - 342s 673ms/step - loss: 0.9568 - accuracy: 0.5697 - val_loss: 0.9690 - val_accuracy: 0.6357
Epoch 7/10
508/508 [==============================] - 344s 678ms/step - loss: 0.9109 - accuracy: 0.5871 - val_loss: 1.1768 - val_accuracy: